# NB-02: Model Modification

**Purpose:** Load MobileMamba-B1, replace the classification head for 6 classes, verify the architecture, and run a forward pass test.

**What changes:**
- Classification head: `BN_Linear(448, 1000)` → `nn.Linear(448, 6)`
- Everything else stays intact

In [4]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../..'))

import torch
import torch.nn as nn

from model.mobilemamba.mobilemamba import MobileMamba, CFG_MobileMamba_B1
from src.model import AnomalyMobileMamba, CLASS_NAMES, NUM_CLASSES

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'Classes: {CLASS_NAMES}')

PyTorch version: 2.6.0+cu124
CUDA available: True
Classes: ['Normal', 'Assault', 'Robbery', 'Shooting', 'Fighting', 'Abuse']


In [5]:
# ─── Load MobileMamba-B1 and replace head ──────────────
PRETRAINED_PATH = '../checkpoints/mobilemamba_b1_pretrained.pth'

model = AnomalyMobileMamba(
    num_classes=NUM_CLASSES,
    pretrained_path=PRETRAINED_PATH if os.path.exists(PRETRAINED_PATH) else None,
)

print('\nModel created successfully.')

[Pretrained] Research Backbone loaded from ../checkpoints/mobilemamba_b1_pretrained.pth
  Missing keys (head expected): 8

Model created successfully.


In [7]:
# ─── Verify Architecture ──────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'Total params:     {total_params / 1e6:.2f}M')
print(f'Trainable params: {trainable_params / 1e6:.2f}M')

print(f'\nHead Structure: {model.backbone.head}')
# Accessing the first Linear layer (0) and the final Linear layer (4)
print(f'  → Input features (Stage 3): {model.backbone.head[0].in_features}')
print(f'  → Hidden features:          {model.backbone.head[0].out_features}')
print(f'  → Output classes:           {model.backbone.head[4].out_features}') # Fix here

# Verify blocks structure
print(f'\nBackbone stages:')
print(f'  blocks1: {len(model.backbone.blocks1)} sub-modules')
print(f'  blocks2: {len(model.backbone.blocks2)} sub-modules')
print(f'  blocks3: {len(model.backbone.blocks3)} sub-modules')


Total params:     16.79M
Trainable params: 16.74M

Head Structure: Sequential(
  (0): Linear(in_features=448, out_features=256, bias=True)
  (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (2): ReLU(inplace=True)
  (3): Dropout(p=0.5, inplace=False)
  (4): Linear(in_features=256, out_features=6, bias=True)
)
  → Input features (Stage 3): 448
  → Hidden features:          256
  → Output classes:           6

Backbone stages:
  blocks1: 2 sub-modules
  blocks2: 6 sub-modules
  blocks3: 5 sub-modules


In [8]:
# ─── Forward Pass Sanity Check ─────────────────────────
model.eval()
dummy = torch.randn(1, 3, 256, 256)  # single frame

with torch.no_grad():
    logits = model(dummy)
    probs = torch.softmax(logits, dim=1)

print(f'Output shape: {logits.shape}')  # expect torch.Size([1, 6])
print(f'Logits:       {logits[0].tolist()}')
print(f'Probabilities:{probs[0].tolist()}')
print(f'Predicted:    {CLASS_NAMES[probs.argmax().item()]}')

assert logits.shape == torch.Size([1, 6]), f'Expected [1, 6], got {logits.shape}'
print('\n✅ Forward pass sanity check PASSED')

Output shape: torch.Size([1, 6])
Logits:       [0.14656531810760498, -0.42861419916152954, 0.048449233174324036, -0.20698949694633484, -0.1339709609746933, 0.1903020739555359]
Probabilities:[0.20114974677562714, 0.11316762119531631, 0.1823510378599167, 0.14124484360218048, 0.15194420516490936, 0.2101426124572754]
Predicted:    Abuse

✅ Forward pass sanity check PASSED


In [9]:
# ─── Test GradCAM ─────────────────────────────────────
dummy = torch.randn(1, 3, 256, 256)
cam, class_idx, probs = model.get_gradcam(dummy)

print(f'GradCAM output shape: {cam.shape}')
print(f'Target class: {class_idx} ({CLASS_NAMES[class_idx]})')
print(f'Probabilities: {probs.tolist()}')
print('\n✅ GradCAM test PASSED')

GradCAM output shape: torch.Size([1, 1, 4, 4])
Target class: 5 (Abuse)
Probabilities: [0.20434872806072235, 0.1120428815484047, 0.1817730814218521, 0.14298605918884277, 0.146079421043396, 0.21276980638504028]

✅ GradCAM test PASSED


In [10]:
# ─── Save Modified Model ──────────────────────────────
save_path = '../checkpoints/mobilemamba_b1_modified.pth'
torch.save({
    'model': model.state_dict(),
    'num_classes': NUM_CLASSES,
}, save_path)

print(f'Modified model saved → {save_path}')
print(f'File size: {os.path.getsize(save_path) / 1e6:.1f} MB')

Modified model saved → ../checkpoints/mobilemamba_b1_modified.pth
File size: 67.8 MB
